# Validate Storage Container
Current storage folder:
- storage/
-    |-----checkpoints/
-    |-----data/


In [ ]:
import sys, os, json
from dotenv import load_dotenv
from pathlib import Path
from ml.dataset import load_cached_paths, download_dataset
from ml.model import model_paths, CHECKPOINT_DIR

load_dotenv('env.env')
STORAGE_DIR = os.getenv('STORAGE_DIR')

In [ ]:
def valStorage():
    if not Path(STORAGE_DIR).exists():
        print(f"Could not find storage at \"{STORAGE_DIR}\"")
        return
    print(f"Found storage at \"{STORAGE_DIR}\"")

    # Dataset
    print("\nChecking for dataset...")
    dataset_paths = load_cached_paths()
    download = input(f"Found {len(dataset_paths)} downloaded scores. Download more? (y/n) ")
    if download == 'y':
        download_dataset(log=print, progress=lambda i, n: None, max_files=1000000)
        dataset_paths = load_cached_paths()
        print(f"Downloaded {len(dataset_paths)} scores.")

    # Models
    print(f"\nChecking for checkpoints in {CHECKPOINT_DIR}...")
    if not CHECKPOINT_DIR.exists():
        print("Checkpoint folder not found.")
        return

    models_found = []
    for subdir in sorted(CHECKPOINT_DIR.iterdir()):
        if subdir.is_dir():
            cfg_path = subdir / "config.json"
            if cfg_path.exists():
                models_found.append((subdir.name, cfg_path))

    # Legacy flat layout fallback
    legacy_cfg = CHECKPOINT_DIR / "config.json"
    if legacy_cfg.exists() and not any(n == "infill" for n, _ in models_found):
        models_found.append(("infill (legacy)", legacy_cfg))

    if not models_found:
        print("No model configs found. Train a model first.")
        return

    print(f"\nFound {len(models_found)} model(s):\n")
    for model_name, cfg_path in models_found:
        try:
            cfg     = json.loads(cfg_path.read_text())
            epoches = cfg.get("epoches", [])
            weights = cfg_path.parent / "model.pt"
            w_size  = f"{weights.stat().st_size / (1024*1024):.1f} MB" if weights.exists() else "not found"

            print(f"  [{model_name}]")
            print(f"    Config:   {cfg_path}")
            print(f"    Weights:  {w_size}")
            print(f"    vocab_size={cfg.get('vocab_size','?')}  "
                  f"d_model={cfg.get('d_model','?')}  "
                  f"n_heads={cfg.get('n_heads','?')}  "
                  f"n_layers={cfg.get('n_layers','?')}  "
                  f"max_seq_len={cfg.get('max_seq_len','?')}")
            print(f"    Epochs trained: {len(epoches)}", end="")
            if epoches:
                best_val        = min(e[1] for e in epoches)
                last_train, last_val = epoches[-1][0], epoches[-1][1]
                print(f"  |  last train={last_train:.4f}  last val={last_val:.4f}  best val={best_val:.4f}", end="")
            print("\n")
        except Exception as e:
            print(f"  [{model_name}]  Error reading config: {e}\n")

valStorage()

### Display Model Epoches

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ── Discover all models that have epoch data ──────────────────────────────────
def _find_model_epochs():
    results = {}
    if not CHECKPOINT_DIR.exists():
        return results
    for subdir in sorted(CHECKPOINT_DIR.iterdir()):
        if subdir.is_dir():
            cfg_path = subdir / "config.json"
            if cfg_path.exists():
                try:
                    epoches = json.loads(cfg_path.read_text()).get("epoches", [])
                    if epoches:
                        results[subdir.name] = epoches
                except Exception:
                    pass
    # Legacy flat config
    legacy = CHECKPOINT_DIR / "config.json"
    if legacy.exists() and "infill" not in results:
        try:
            epoches = json.loads(legacy.read_text()).get("epoches", [])
            if epoches:
                results["infill (legacy)"] = epoches
        except Exception:
            pass
    return results

models = _find_model_epochs()

if not models:
    print("No epoch data found — train a model first.")
else:
    COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    # One checkbox per model, paired with its plot color
    checkboxes = {
        name: (
            widgets.Checkbox(
                value=True,
                description=name,
                style={"description_width": "initial"},
                layout=widgets.Layout(min_width="160px"),
            ),
            COLORS[i % len(COLORS)],
        )
        for i, name in enumerate(models)
    }

    out = widgets.Output()

    def draw(change=None):
        with out:
            out.clear_output(wait=True)
            visible = [(n, e) for n, e in models.items() if checkboxes[n][0].value]
            if not visible:
                print("No models selected — tick a checkbox to show it.")
                return

            fig, ax = plt.subplots(figsize=(11, 5))
            for name, epoches in visible:
                _, color = checkboxes[name]
                xs = range(1, len(epoches) + 1)
                ax.plot(xs, [e[0] for e in epoches], color=color, linestyle="-",  label=f"{name} — train")
                ax.plot(xs, [e[1] for e in epoches], color=color, linestyle="--", label=f"{name} — val")

            ax.set_xlabel("Epoch")
            ax.set_ylabel("Loss")
            ax.set_title("Training vs Validation Loss")
            ax.legend(loc="upper right", fontsize=8, ncol=max(1, len(visible) // 4))
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

    for name in models:
        checkboxes[name][0].observe(draw, names="value")

    # Show All / Hide All buttons
    btn_all  = widgets.Button(description="Show All", button_style="info",    layout=widgets.Layout(width="100px"))
    btn_none = widgets.Button(description="Hide All", button_style="warning", layout=widgets.Layout(width="100px"))
    btn_all .on_click(lambda _: [checkboxes[n][0].__setattr__("value", True)  for n in models])
    btn_none.on_click(lambda _: [checkboxes[n][0].__setattr__("value", False) for n in models])

    controls = widgets.VBox([
        widgets.HBox([btn_all, btn_none], layout=widgets.Layout(gap="8px")),
        widgets.HBox([cb for cb, _ in checkboxes.values()], layout=widgets.Layout(flex_flow="row wrap", gap="4px 16px")),
    ])

    display(widgets.VBox([controls, out]))
    draw()